# Week 6, Day 1: Spatial Operations

## Today

By the end of this session you be able to:

- Write a SQL where clause for `arcpy.management.SelectLayerByAttribute`
- Chain an attribute selection into a spatial one with `arcpy.management.SelectLayerByLocation` (`WITHIN_A_DISTANCE`, `INTERSECT`)
- Count a selection with `arcpy.management.GetCount` 
- Read the `SHAPE@AREA` and `SHAPE@LENGTH` geometry tokens, and explain why the values they return depend on a layer's coordinate system.



## Questions that are selections

Many of the questions a GIS analyst answers are selections: 
- which parks are within 3 miles of a municipality? 
- Which streams run through Kent? 
- Which parcels are zoned residential **and** lie within a floodplain? 

Each question has essentially the same structure... Narrow a layer by an **attribute** test (a value in a field), by a **spatial** test (its relationship to another layer's geometry), or by both in sequence.


**Predict before you read on:** `arcpy.analysis.Buffer`, the arcpy tool we've been using extensive accepts a shapefile path as its input with no setup. 

The first parameter of `arcpy.management.SelectLayerByAttribute` is named `in_layer_or_view`, not `in_features`. 

Is that only a naming difference, or does the tool require something other than a path such as `"portage_munis.shp"` before it can select anything?

## Layers hold a selection, not datasets

A shapefile on disk is static. It has no notion of currently selected rows. 

In contrast, a **layer** does: it is a reference to a dataset together with (among other properties) the subset of rows that is currently selected. THAT is what the `in_layer_or_view` parameter asks for.

In current versions of arcpy, `SelectLayerByAttribute` also accepts a dataset path directly. Given `"portage_munis.shp"`, it creates a temporary layer, runs the selection, and returns the result, so (strictly speaking) the answer to the prediction is NO. 

But a temporary layer has no name, so it cannot be passed to a second tool call, and the chained selections in this session require exactly that. 

The solution is `arcpy.management.MakeFeatureLayer(in_features, out_layer)`: create one named layer per dataset at the start, and every later selection refers to it by name.


## Let's start

As usual, setup you workspace


In [ ]:
import arcpy

arcpy.env.workspace = "C:/YOURPATHHERE/data/part1"
arcpy.env.overwriteOutput = True


**Expected result:** as always nothing prints

### One conversion first

`Describe` can read `streams_portage.geojson`, and `ListFeatureClasses` ignores it. The layer and selection tools in this session are stricter still: `MakeFeatureLayer` will not accept a GeoJSON file at all. It needs a feature class, which in a folder workspace means a shapefile.

The fix is one tool call, run once. `arcpy.conversion.JSONToFeatures(in_json_file, out_features, geometry_type)` reads a GeoJSON file and writes a feature class. The third argument is required for GeoJSON input, because a GeoJSON file may mix geometry types and a feature class cannot; this file holds lines, so the value is `"POLYLINE"`.


In [ ]:
arcpy.conversion.JSONToFeatures("streams_portage.geojson", "streams_portage.shp", "POLYLINE")


**Expected result:** nothing prints, and `streams_portage.shp` (with its sidecar files) now exists in the workspace, holding the same 1,989 segments as the GeoJSON. Every reference to the streams from here on uses the shapefile.


In [ ]:
arcpy.management.MakeFeatureLayer("portage_adj_munis.shp", "munis_lyr")
arcpy.management.MakeFeatureLayer("ohio_state_parks.shp", "parks_lyr")
arcpy.management.MakeFeatureLayer("streams_portage.shp", "streams_lyr")

**Expected result:** nothing prints, but three named layers now exist in this notebook's session: 
1. `munis_lyr`
2. `parks_lyr`
3. `streams_lyr`

They reference `portage_adj_munis.shp`, `ohio_state_parks.shp`, and the `streams_portage.shp` you just converted. None has a selection yet.

## `SelectLayerByAttribute`: a SQL where clause

`SelectLayerByAttribute(in_layer_or_view, selection_type, where_clause)` narrows a layer's selection with a SQL expression on its attribute fields, the same kind of expression used in Pro's Select By Attributes dialog. 

For a shapefile, or a layer built from one, the quoting rule is: **field names in double quotes, text values in single quotes, numeric values unquoted.** 

`"NAME" = 'Kent'` and `"ALAND" > 10000000` both follow it. 

Writing `NAME = 'Kent'` without the double quotes around the field name is a common cause of a SQL expression error on shapefile data. 

(FYI, Enterprise geodatabases use different delimiters. `arcpy.AddFieldDelimiters()` builds the field-name part portably; it is not needed for this session's data.)

`portage_munis.shp` has no population field. It is TIGER Place data, and its fields are `NAME`, `NAMELSAD`, `ALAND`, `AWATER`, and several Census identifiers. `ALAND`, land area in square meters, is the closest available measure of a place's size, so the attribute query uses it. `"ALAND" > 10000000` selects municipalities with more than 10,000,000 square meters of land, about 3.9 square miles. That separates the larger places in the seven-county region from the small villages in the same layer.


**Predict before you run:** `portage_munis.shp` contains 186 municipalities, with land areas ranging from under 200,000 square meters for the smallest villages to over 200 million for Cleveland. Does `"ALAND" > 10000000` select about a dozen of the 186, about half, or nearly all?

In [ ]:
arcpy.management.SelectLayerByAttribute("munis_lyr", "NEW_SELECTION", '"ALAND" > 10000000')

muni_count = arcpy.management.GetCount("munis_lyr")
print(f"municipalities selected: {muni_count}")

**Expected result:** `municipalities selected: 87`, just under half of the 186. `GetCount` returns a `Result` object rather than a number, but an f-string prints it as one. 

A count in a variable is one you can compare, log, or write into a report.

## `SelectLayerByLocation`: a spatial test between two layers

Now the spatial question: which parks lie within 3 miles of one of those 87 municipalities? `SelectLayerByLocation(in_layer, overlap_type, select_features, search_distance, selection_type)` tests every feature in `in_layer` against `select_features` and keeps the ones that pass. 

`overlap_type="WITHIN_A_DISTANCE"` with `search_distance="3 Miles"` expresses the relationship this question needs.

***NOTE:*** `select_features` is not tested against every row in `munis_lyr`'s dataset. 

It is tested against whatever is **currently selected** in `munis_lyr`, which is the 87-municipality subset from the previous cell, not all 186. 

This is what a chained attribute and spatial selection means in practice: run the attribute selection first, then pass the same layer to a spatial tool as `select_features`. The spatial test uses the narrowed selection.


**Predict before you run:** Ohio's 67 state parks are distributed statewide, and most are not in northeast Ohio. Will "within 3 miles of one of the 87 larger municipalities" select several parks, or only one or two?

In [ ]:
arcpy.management.SelectLayerByLocation(
    "parks_lyr", "WITHIN_A_DISTANCE", "munis_lyr", "3 Miles", "NEW_SELECTION"
)

park_count = arcpy.management.GetCount("parks_lyr")
print(f"parks within 3 miles of a larger municipality: {park_count}")

**Expected result:** `parks within 3 miles of a larger municipality: 6`: Mosquito Lake, Portage Lakes, Tinkers Creek, Kent Stark, West Branch, and Wingfoot Lake State Parks. 

Testing against all 186 municipalities instead of the 87 would select 9; Lake Milton, Nelson-Kennedy Ledges, and Punderson State Parks would also qualify. 

Those three are excluded here because the municipalities nearest them did not pass the `ALAND` test. 

**Your turn:** Rerun the parks selection with `search_distance` set to `"1 Mile"` instead of `"3 Miles"`, keeping the same `munis_lyr` selection (the 87 larger municipalities) and the same `overlap_type`. Predict first: does the smaller radius drop some of the six parks, or do all six remain?



In [ ]:
## Try it here



## A second question: which streams cross Kent?

`SelectLayerByAttribute` with `selection_type="NEW_SELECTION"` replaces whatever the layer had selected (rather than adding to it). 

That makes `munis_lyr` easy to reuse for a different question: 
- select it down to one place
- then pass it to `streams_lyr` as `select_features` for an `INTERSECT` test

`INTERSECT` asks only whether the geometries touch or cross, so `search_distance` is not needed. Skip it by passing `selection_type` as a keyword argument instead of filling the next positional slot


**Predict before you run:** `streams_portage.geojson` contains 1,989 stream, ditch, river, and canal segments within Portage County. Will the number of segments that cross Kent be in the single digits, the dozens, or the hundreds?

In [ ]:
arcpy.management.SelectLayerByAttribute("munis_lyr", "NEW_SELECTION", "\"NAME\" = 'Kent'")

arcpy.management.SelectLayerByLocation(
    "streams_lyr", "INTERSECT", "munis_lyr", selection_type="NEW_SELECTION"
)

stream_count = arcpy.management.GetCount("streams_lyr")
print(f"stream segments crossing Kent: {stream_count}")

**Expected result:** `stream segments crossing Kent: 102` out of 1,989. Portage County's stream network is dense and Kent is not a small place. 

The `"NAME" = 'Kent'` call replaced the earlier 87-municipality selection with a single match. If it used `"ADD_TO_SELECTION"` instead of `"NEW_SELECTION"`, Kent would have joined the 87, and the intersect test would have run against 88 municipalities rather than Kent alone.

**Your turn:** Rerun the `"ALAND" > 10000000` cell to restore the 87-municipality selection on `munis_lyr`. 

Then change `"NEW_SELECTION"` to `"ADD_TO_SELECTION"` on the Kent attribute call in the cell above and rerun that cell. 

Kent will join the 87 instead of replacing them, and the `INTERSECT` test in the same cell will use whichever selection is current when it runs. 

Predict first: does `stream_count` still read 102, or does testing against 88 municipalities change it?


In [ ]:
## Try it here



## Let's try something else


**Predict before you run:** The attribute tests so far have used one field, `ALAND`. 

`NAMELSAD` is different: it includes each place's Census legal designation in the name, as in `"Kent city"`, `"Mantua village"`, and `"Suffield CDP"` (Census Designated Place, an unincorporated community the Census tracks). 

A `LIKE` where clause matches a pattern rather than an exact value, with `%` standing for any characters. 

For example, `"NAMELSAD" LIKE '%city'` matches every name ending in `city`. 

The quoting rule is the same as before: field name in double quotes, pattern in single quotes. Of 186 municipalities, how many have legal city status: about a dozen, about forty, or more than seventy?

In [ ]:
arcpy.management.SelectLayerByAttribute("munis_lyr", "NEW_SELECTION", "\"NAMELSAD\" LIKE '%city'")

city_count = arcpy.management.GetCount("munis_lyr")
print(f"municipalities with city status: {city_count}")

**What did we get?**

## Geometry tokens and coordinate systems

When we used cursors, wen read `SHAPE@AREA` from `ohio_counties.shp` and found it reporting square degrees rather than square meters, because that shapefile is in a geographic coordinate system. 

`portage_munis.shp` is in the same coordinate system (NAD83, EPSG:4269), so the same problem applies here, and `SHAPE@LENGTH` behaves the same way for line data such as `streams_portage.geojson`. 

Whatever unit a layer's coordinate system uses is the unit `SHAPE@AREA` and `SHAPE@LENGTH` return, degrees included. 

When a real area or length matters, there are two options: 
1. project a copy of the data into a projected coordinate system first
2. use a precomputed attribute field when one exists, such as `ALAND`, which the Census stores in square meters regardless of the shapefile's coordinate system. Of course, you need to know the CRS and trust the dataset


**Predict before you run:** `portage_munis.shp` is in the same NAD83 geographic coordinate system as `ohio_counties.shp`. Will `SHAPE@AREA` print numbers on the same scale as `ALAND`'s square-meter values, or much smaller ones?

In [ ]:
with arcpy.da.SearchCursor("portage_munis.shp", ["NAME", "SHAPE@AREA", "ALAND"]) as cursor:
    for i, row in enumerate(cursor):
        if i >= 3:
            break
        name, shape_area, aland = row
        print(f"{name:<15} SHAPE@AREA={shape_area:.6f} sq deg   ALAND={aland:,.0f} sq m")

**Expected result:** the first three rows in the shapefile's own order (not sorted):

```
East Sparta     SHAPE@AREA=0.000505 sq deg   ALAND=4,740,210 sq m
North Industry  SHAPE@AREA=0.000546 sq deg   ALAND=5,125,254 sq m
Reedurban       SHAPE@AREA=0.000524 sq deg   ALAND=4,910,214 sq m
```

`SHAPE@AREA` here is a fraction of one square degree, which is not a usable real-world size, while `ALAND` is a genuine square-meter value.

 Both describe the same municipality; only one is meaningful without a projection step.

## Paired activity: practicum prep

Find a buddy

Here's your question:

> "Which of Portage County's municipalities are within 3 miles of a state park but don't have one inside their own boundaries? Find the muni names and the TOTAL land area involved."

Everything you need is in this notebook: `ohio_counties.shp`, `portage_munis.shp`, `ohio_state_parks.shp`, the tools from today, and a cursor.

**Before you write any code**, take five minutes on paper and agree on three things:

1. What does "Portage County's municipalities" mean in this data? `portage_munis.shp` covers seven counties, and some municipalities cross the county line. Decide how you will select the Portage ones, and write down why
2. What is the sequence of selections? Each step narrows or trims the set from the step before. Write the steps in order, in plain language
3. What comes out at the end, and in what units?

**Then build it.** Two things you have not done yet today:

- Once you have a selection on a layer, the next `SelectLayerByLocation` call should work on that selection rather than starting over. `selection_type` has more values than the two used above. Open the tool's reference page and find the ones that narrow an existing selection, and the one that removes from it.
- For the names and the land area, run a `SearchCursor` on the *layer*, not on the shapefile. A cursor on a layer walks only the selected features. Report the total from `ALAND`, converted to acres, and say why you did not use `SHAPE@AREA`.

**Finally, compare your answer with another group**


In [ ]:
pass  # your code here
